# What Drives the Price of a Car?

**Data Science Lifecycle (MSDS-1) · Advent 2026 · Assignment One**
Uganda Christian University, Faculty of Engineering, Design and Technology

**Name:** Tina Jacqueline

---

The brief is to use `automobileEDA.csv` to work out which features actually move the price of a
car. The notebook runs top to bottom: Section 1 cleans the data and writes `Ssekanabi_Data.csv`,
Sections 2 to 4 describe and test the relationships, and Section 5 pulls everything together into
a ranked list of price drivers and some practical advice for a dealer.

Every output below has a short written interpretation underneath it.

## 0. Setup

Nothing interesting here, just imports and the settings used for the rest of the notebook.
Figures are written to `figures/` as they are produced so they can be reused in the report.

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

LAST_NAME = "SSEKANABI"
RAW_PATH = Path("automobileEDA.csv")
CLEAN_PATH = Path(f"{LAST_NAME}_Data.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})


def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png")
    plt.show()


print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

pandas 3.0.6 | numpy 2.5.3 | seaborn 0.13.2


---

# 1. Exploring and cleaning the data

> Explore the features in the dataset that could be used to predict the price of a car. Check for
> missing values, duplicates and outliers, and handle any you find. Save the cleaned dataset as
> `Ssekanabi_Data.csv`.

### 1.1 First look

In [226]:
df = pd.read_csv(RAW_PATH)
print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df.head()

201 rows, 29 columns


,symboling,normalized-losses,make,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,width,height,curb-weight,engine-type,num-of-cylinders,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,city-L/100km,horsepower-binned,diesel,gas
0,3,122,alfa-romero,std,two,convertible,rwd,front,88.600,0.811,0.890,48.800,2548,dohc,four,130,mpfi,3.470,2.680,9.000,111.000,"5,000.000",21,27,13495,11.190,Medium,0,1
1,3,122,alfa-romero,std,two,convertible,rwd,front,88.600,0.811,0.890,48.800,2548,dohc,four,130,mpfi,3.470,2.680,9.000,111.000,"5,000.000",21,27,16500,11.190,Medium,0,1
2,1,122,alfa-romero,std,two,hatchback,rwd,front,94.500,0.823,0.910,52.400,2823,ohcv,six,152,mpfi,2.680,3.470,9.000,154.000,"5,000.000",19,26,16500,12.368,Medium,0,1
3,2,164,audi,std,four,sedan,fwd,front,99.800,0.849,0.919,54.300,2337,ohc,four,109,mpfi,3.190,3.400,10.000,102.000,"5,500.000",24,30,13950,9.792,Medium,0,1
4,2,164,audi,std,four,sedan,4wd,front,99.400,0.849,0.922,54.300,2824,ohc,five,136,mpfi,3.190,3.400,8.000,115.000,"5,500.000",18,22,17450,13.056,Medium,0,1


201 cars and 29 columns. The file name is a hint: this is not the raw UCI automobile data, it is a
version somebody has already partly cleaned. That turns out to matter, and most of Section 1 is me
finding the traces they left behind.

In [227]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          201 non-null    int64  
 1   normalized-losses  201 non-null    int64  
 2   make               201 non-null    str    
 3   aspiration         201 non-null    str    
 4   num-of-doors       201 non-null    str    
 5   body-style         201 non-null    str    
 6   drive-wheels       201 non-null    str    
 7   engine-location    201 non-null    str    
 8   wheel-base         201 non-null    float64
 9   length             201 non-null    float64
 10  width              201 non-null    float64
 11  height             201 non-null    float64
 12  curb-weight        201 non-null    int64  
 13  engine-type        201 non-null    str    
 14  num-of-cylinders   201 non-null    str    
 15  engine-size        201 non-null    int64  
 16  fuel-system        201 non-null    st

Only `stroke` and `horsepower-binned` report missing values, and everything else is complete. That
is suspicious for a dataset of this vintage. The original UCI file has gaps in
`normalized-losses`, `bore`, `stroke`, `horsepower`, `peak-rpm`, `num-of-doors` and `price`, so
somebody has filled those in already. Subsection 1.4 is about finding where.

### 1.2 What the columns are, and which of them could predict price

In [228]:
# Splitting by role rather than by dtype, because a few numeric-looking columns
# (symboling, diesel, gas) are really categorical.
CATEGORICAL = ["make", "aspiration", "num-of-doors", "body-style", "drive-wheels",
               "engine-location", "engine-type", "num-of-cylinders", "fuel-system",
               "horsepower-binned"]

CONTINUOUS = ["wheel-base", "length", "width", "height", "curb-weight", "engine-size",
              "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
              "city-mpg", "highway-mpg", "city-L/100km", "normalized-losses"]

OTHER = ["symboling", "diesel", "gas", "price"]

assert set(CATEGORICAL + CONTINUOUS + OTHER) == set(df.columns)
print(f"{len(CONTINUOUS)} continuous, {len(CATEGORICAL)} categorical, {len(OTHER)} other")

15 continuous, 10 categorical, 4 other


Grouping them by what they actually describe makes the structure clearer:

| Group | Columns | Why it might predict price |
|---|---|---|
| Size and mass | `wheel-base`, `length`, `width`, `height`, `curb-weight` | Bigger cars cost more to build, and size is most of what separates a hatchback from a saloon |
| Engine | `engine-size`, `num-of-cylinders`, `engine-type`, `bore`, `stroke`, `compression-ratio`, `fuel-system` | The engine is usually the most expensive single assembly in the car |
| Performance | `horsepower`, `peak-rpm`, `horsepower-binned`, `aspiration` | What a buyer is knowingly paying a premium for |
| Economy | `city-mpg`, `highway-mpg`, `city-L/100km` | Inversely related to engine size, so likely a proxy rather than a cause |
| Layout and body | `drive-wheels`, `engine-location`, `body-style`, `num-of-doors` | Rear-wheel drive and rear engines mark out the premium and sports segments |
| Brand | `make` | Brand equity gets priced directly |
| Risk | `symboling`, `normalized-losses` | Insurance risk rating, related to the car's class but not obviously to its price |
| Fuel | `diesel`, `gas` | One-hot flags for fuel type |

`price` is the target. `symboling` is an insurance risk score running from -3 to +3, so although it
is stored as an integer it is really ordinal. My expectation going in is that the size, engine and
performance groups do the heavy lifting, and that `city-L/100km` adds nothing over `city-mpg`
because of the way it is defined.

### 1.3 Missing values

In [229]:
missing = df.isna().sum()
missing[missing > 0].to_frame("n_missing").assign(
    pct=lambda d: (100 * d.n_missing / len(df)).round(2))

,n_missing,pct
stroke,4,1.990
horsepower-binned,1,0.500


Five missing cells out of 5,829. Small enough to look at individually instead of reaching straight
for `fillna`.

In [230]:
df[df["stroke"].isna()][["make", "engine-type", "num-of-cylinders", "engine-size",
                         "bore", "stroke", "price"]]

,make,engine-type,num-of-cylinders,engine-size,bore,stroke,price
52,mazda,rotor,two,70,3.330,NaN,10945
53,mazda,rotor,two,70,3.330,NaN,11845
54,mazda,rotor,two,70,3.330,NaN,13645
55,mazda,rotor,two,80,3.330,NaN,15645


All four are Mazda rotaries. This is not a data entry failure, it is a structural absence: a Wankel
rotary has no pistons, so bore and stroke are not defined for it. The `bore` value of 3.3298 on
those same rows is itself a mean fill, which comes up again in 1.4.

Three options: drop the column, drop the four rows, or impute. Dropping four cars to save one weak
column is a bad trade, and these are the only rotaries in the dataset, so they carry information no
other row has. `stroke` correlates with price at about 0.08, so whatever goes in those cells will
not change any conclusion. I fill with the median and note it.

In [231]:
stroke_median = df["stroke"].median()
df["stroke"] = df["stroke"].fillna(stroke_median)
print(f"stroke: 4 rotary engines filled with the median, {stroke_median}")

stroke: 4 rotary engines filled with the median, 3.29


In [232]:
df[df["horsepower-binned"].isna()][["make", "num-of-cylinders", "engine-size",
                                    "horsepower", "horsepower-binned", "price"]]

,make,num-of-cylinders,engine-size,horsepower,horsepower-binned,price
46,jaguar,twelve,326,262.000,NaN,36000


The one missing bin belongs to the V12 Jaguar, the most powerful car in the set at 262 hp.
`horsepower-binned` is not data, it is a transformation of `horsepower`, so imputing it would be
the wrong move. The right move is to work out which bins were used and apply them consistently.

In [233]:
df.groupby("horsepower-binned", observed=True)["horsepower"].agg(["min", "max", "count"])

,min,max,count
horsepower-binned,,,
High,156.000,207.000,23
Low,48.000,101.000,115
Medium,102.000,155.000,62


Low runs 48 to 101, Medium 102 to 155, High 156 to 207. Those are three equal-width bins across the
range 48 to 207, which means the Jaguar's 262 hp was excluded when the bins were cut and then fell
outside the top edge. Rebuilding the bins with an open upper bound fixes it and leaves every other
row exactly where it was.

In [ ]:
df["horsepower-binned"] = pd.cut(df["horsepower"],
                                 bins=[0, 101, 155, np.inf],
                                 labels=["Low", "Medium", "High"])

print("missing after rebinning:", df["horsepower-binned"].isna().sum())
df.groupby("horsepower-binned", observed=True)["horsepower"].agg(["min", "max", "count"])

Nothing missing now, and the only change is the Jaguar moving into High where it belongs. High has
24 cars instead of 23.

### 1.4 Values that were filled in before I got the file

This is the part `isna()` will not show. If an earlier step replaced missing values with the column
mean, those cells now look like ordinary data. The giveaway is a value repeating far more often
than it reasonably should.

In [ ]:
df["normalized-losses"].value_counts().head(5)

122 appears 41 times, and the next most common value appears 11 times. 122 is also close to the
mean of the remaining values, and the original UCI file is missing `normalized-losses` on exactly
41 rows.

So a fifth of this column is a constant that was invented. That is not wrong in itself, but it does
mean any correlation between `normalized-losses` and price is diluted by 20% of the rows sitting on
a flat line. I keep the column and treat whatever it reports with suspicion rather than drop it.

In [ ]:
# Horsepower and peak-rpm are stored as floats. Any row that is not a whole
# number has to be a fill rather than a measurement.
df[df["horsepower"] % 1 != 0][["make", "body-style", "horsepower", "peak-rpm", "price"]]

Two Renaults, both carrying 104.256 hp and 5125.37 rpm, which are the column means. The same story
as `normalized-losses` on a much smaller scale. Two rows out of 201 will not distort anything, so I
round both columns back to integers, which is how horsepower and rpm are actually recorded.

In [ ]:
df["horsepower"] = df["horsepower"].round().astype(int)
df["peak-rpm"] = df["peak-rpm"].round().astype(int)
df[["horsepower", "peak-rpm"]].dtypes

### 1.5 Duplicates

In [ ]:
print("fully identical rows:", df.duplicated().sum())

None, which is the easy answer. The more useful question is whether any two cars match on every
specification and differ only in price, because that would be either a double entry or something
genuinely interesting.

In [ ]:
spec_cols = [c for c in df.columns if c != "price"]
pairs = df[df.duplicated(spec_cols, keep=False)].sort_values(spec_cols)
pairs[["make", "body-style", "drive-wheels", "engine-size", "horsepower",
       "curb-weight", "price"]]

Seven pairs, fourteen rows. Look at the prices. The Alfa Romeo convertibles are 13,495 and 16,500,
the Porsche hardtops 32,528 and 34,028, the Mazda saloons 8,495 and 10,245. These are trim levels:
the same mechanical car sold with different equipment at a different price. That is exactly the
kind of record this dataset should contain, and deleting one of each pair would throw away real
variation in price at a fixed specification.

I keep all fourteen. Worth remembering later that price is not a deterministic function of the
specification columns, because here are seven pairs that prove it.

### 1.6 Outliers

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < lo) | (s > hi)
    return pd.Series({"n_outliers": int(mask.sum()), "lower": lo, "upper": hi,
                      "min": s.min(), "max": s.max()})


numeric_check = ["wheel-base", "length", "width", "height", "curb-weight", "engine-size",
                 "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
                 "city-mpg", "highway-mpg", "normalized-losses", "price"]

pd.DataFrame({c: iqr_outliers(df[c]) for c in numeric_check}).T \
  .sort_values("n_outliers", ascending=False)

The Tukey rule flags 27 values in `compression-ratio`, 20 in `stroke`, 14 in `price` and 11 in
`width`. That is a lot, and it is where a mechanical application of the rule would do real damage.
Before removing anything I want to see what the flagged rows actually are.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, ["price", "engine-size", "horsepower",
                               "compression-ratio", "curb-weight", "stroke"]):
    sns.boxplot(x=df[col], ax=ax, color="#4C72B0", width=0.5)
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Where the IQR rule flags values", y=1.01, fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig("01_outlier_boxplots")

In [ ]:
q1, q3 = df["price"].quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
print(f"price upper fence: {upper_fence:,.0f}")

df[df["price"] > upper_fence][["make", "body-style", "num-of-cylinders", "engine-size",
                               "horsepower", "price"]].sort_values("price")

Every one of the fourteen is a BMW, Mercedes-Benz, Jaguar or Porsche. They are not errors, they are
the luxury end of the market, and they are the most informative rows in the dataset for a question
about what makes cars expensive. Removing them would compress the price range from 5,118 to 45,400
down to roughly 5,118 to 29,588 and would weaken precisely the relationships I am trying to
measure. They stay.

In [ ]:
df.groupby("diesel")["compression-ratio"].agg(["count", "min", "median", "max"]) \
  .rename(index={0: "petrol", 1: "diesel"})

And there is the explanation for `compression-ratio`. The 181 petrol cars sit between 7.0 and 11.5;
the 20 diesels sit between 21.0 and 23.0. The column is bimodal by design, because that is how
compression ignition works. The IQR rule assumes a single roughly symmetric distribution, so it
flags the entire diesel population as outliers. Treating them as errors would be a serious mistake.

`stroke` is the same problem in a milder form, with the flat-four Subarus clustering near 2.64
against a general median of 3.29.

**Decision on outliers.** I detect and document them but remove none. Every value flagged here is
either a genuine luxury car, a diesel engine or a particular engine layout. The one concession I
make is to report the median alongside the mean whenever I summarise price, because the
distribution is clearly right-skewed.

### 1.7 Columns that repeat each other

In [ ]:
# city-L/100km is meant to be the metric conversion of city-mpg.
print("max |235/city-mpg - city-L/100km| =",
      (235 / df["city-mpg"] - df["city-L/100km"]).abs().max())

# diesel and gas are one-hot flags for the same underlying variable.
print("rows where diesel + gas == 1:", (df["diesel"] + df["gas"] == 1).sum(), "of", len(df))

`city-L/100km` is 235 divided by `city-mpg` to within floating point noise, so it carries nothing
that `city-mpg` does not. And `diesel` and `gas` sum to 1 on every row, which is the dummy variable
trap: either one is a perfect linear function of the other.

I keep both in the saved file, since the assignment asks me to clean the dataset rather than
redesign it, but I drop `city-L/100km` and `gas` when I build the modelling matrix in Section 5.
Leaving them in would let a model split one effect across two columns and make the importance
scores meaningless.

In [ ]:
df[["wheel-base", "length", "width", "height"]].agg(["min", "max"]).T

One more thing to flag before Section 2. `length` and `width` have been scaled, `wheel-base` and
`height` have not. The minimum of `length` is 0.678 rather than 0, so this is not min-max
normalisation, it is each value divided by the column maximum. It makes no difference to
correlations or model importances, but it does mean the `length` axis on any chart cannot be read
as inches.

### 1.8 Saving the cleaned dataset

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
df.to_csv(Path("data/processed") / CLEAN_PATH.name, index=False)

check = pd.read_csv(CLEAN_PATH)
print(f"saved {CLEAN_PATH}  ->  {check.shape[0]} rows x {check.shape[1]} columns")
print("missing values remaining:", int(check.isna().sum().sum()))
print("duplicate rows:", int(check.duplicated().sum()))

**What Section 1 changed**

| Issue found | What I did |
|---|---|
| 4 missing `stroke`, all Mazda rotaries | Filled with the median. Rotaries have no stroke, and the column barely predicts price |
| 1 missing `horsepower-binned`, the V12 Jaguar | Rebuilt the bins with an open upper edge instead of imputing a derived column |
| `normalized-losses` = 122 on 41 rows | Left alone, flagged as an upstream mean fill covering 20% of the column |
| `horsepower` and `peak-rpm` means on 2 Renaults | Rounded both columns to integers |
| 0 exact duplicates, 7 spec-identical pairs | Kept. Trim variants, not double entries |
| 14 price outliers, 27 compression-ratio, 20 stroke | Kept. Luxury cars, diesel engines and flat-fours respectively |
| `city-L/100km` duplicates `city-mpg`, `diesel` + `gas` = 1 | Kept in the file, excluded from the Section 5 model matrix |

The row count is unchanged at 201, and I think that is the right outcome. The dataset arrived in
decent shape, and the useful work was identifying what had already been done to it rather than
deleting things.

---

# 2. Descriptive statistics and patterns &nbsp; *[5 Marks]*

> Analyse patterns and run descriptive statistics on the dataset. Give summary statistics for the
> continuous variables and frequency counts for the categorical variables. Use graphs to show how
> price and the other variables are distributed, and summarise the findings on the patterns
> observed.

### 2.1 Summary statistics for the continuous variables

In [ ]:
stats_table = df[CONTINUOUS + ["price"]].describe().T
stats_table["skew"] = df[CONTINUOUS + ["price"]].skew()
stats_table["kurtosis"] = df[CONTINUOUS + ["price"]].kurt()
stats_table[["count", "mean", "std", "min", "50%", "max", "skew", "kurtosis"]].round(2)

A few things stand out.

`price` has a mean of 13,207 against a median of 10,295, with a skew of 1.81. That gap of nearly
3,000 is the luxury tail pulling the average up, and it is the reason I use the median whenever I
describe a typical car.

`compression-ratio` has a skew of 2.6 and a standard deviation of 4.0 on a mean of 10.2, which
looks alarming until you remember from 1.6 that it is two distributions stacked on top of each
other.

`peak-rpm` and `height` are nearly symmetric. `normalized-losses` has a suspiciously tight middle,
which is the 41 rows pinned at 122.

### 2.2 Frequency counts for the categorical variables

In [ ]:
for col in CATEGORICAL:
    counts = df[col].value_counts()
    print(f"--- {col}  ({counts.size} levels) ---")
    print(counts.to_string())
    print()

The counts matter more than they look, because several of these levels are far too thin to support
a conclusion:

- `engine-location` is 198 front against **3** rear, and all three rear-engined cars are Porsches.
  Any mean price for that level is really the mean price of three Porsches.
- `num-of-cylinders` has single cars in the three-cylinder and twelve-cylinder levels.
- `fuel-system` has levels `spfi` and `mfi` with one car each.
- `body-style` has 6 convertibles and 8 hardtops against 94 saloons and 68 hatchbacks.
- `make` spreads 201 cars across 22 brands, so the median brand has about 8 cars.

`drive-wheels`, `aspiration`, `num-of-doors` and `horsepower-binned` are the only categoricals with
reasonably populated levels throughout. I keep the thin ones in the analysis but I will not draw
conclusions from them without saying so.

### 2.3 How price is distributed

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(df["price"], bins=30, kde=True, ax=axes[0], color="#4C72B0")
axes[0].axvline(df["price"].mean(), color="#C44E52", ls="--", label=f"mean {df['price'].mean():,.0f}")
axes[0].axvline(df["price"].median(), color="#55A868", ls="--", label=f"median {df['price'].median():,.0f}")
axes[0].set_title("Price")
axes[0].legend()

sns.boxplot(x=df["price"], ax=axes[1], color="#4C72B0", width=0.4)
axes[1].set_title("Price, boxplot")

sns.histplot(np.log(df["price"]), bins=30, kde=True, ax=axes[2], color="#8172B3")
axes[2].set_title("log(Price)")

plt.tight_layout()
save_fig("02_price_distribution")

print(f"skew of price      : {df['price'].skew():.3f}")
print(f"skew of log(price) : {np.log(df['price']).skew():.3f}")

Clearly right-skewed, and unimodal with a long thin tail rather than a second cluster. Most of the
market sits between 5,000 and 20,000, and above that the cars thin out into the luxury brands
identified in 1.6.

Taking logs pulls the skew from 1.81 down to 0.68, which is nearly symmetric. That is a useful
thing to know if this moves on to regression later, because a linear model on raw price will spend
most of its effort fitting the handful of expensive cars. I keep working with raw price here
because the assignment is about interpretation rather than prediction, and raw shillings are easier
to talk about.

### 2.4 How the other continuous variables are distributed

In [ ]:
plot_cols = ["wheel-base", "length", "width", "height", "curb-weight", "engine-size",
             "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm", "highway-mpg"]

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, col in zip(axes.flat, plot_cols):
    sns.histplot(df[col], bins=25, ax=ax, color="#4C72B0")
    ax.set_title(col, fontsize=11)
    ax.set_ylabel("")
plt.tight_layout()
save_fig("03_continuous_distributions")

`engine-size`, `horsepower` and `curb-weight` all have the same right-skewed shape as price, which
is an early sign that they track it. `compression-ratio` shows the two-population split plainly:
a dense lump around 9 and a detached group above 21.

`peak-rpm` is lumpy rather than smooth because manufacturers quote round numbers, so the values
pile up at 4,800, 5,000, 5,200 and 5,500. `width` is tightly concentrated, which is unsurprising
given that road vehicles have to fit in a lane.

### 2.5 How the categorical variables are distributed

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, ["body-style", "drive-wheels", "num-of-cylinders",
                               "engine-type", "fuel-system", "aspiration"]):
    order = df[col].value_counts().index
    sns.countplot(y=df[col], order=order, ax=ax, color="#4C72B0")
    ax.set_title(col, fontsize=11)
    ax.set_ylabel("")
plt.tight_layout()
save_fig("04_categorical_counts")

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(y=df["make"], order=df["make"].value_counts().index, color="#4C72B0")
plt.title("Cars per make")
plt.ylabel("")
save_fig("05_make_counts")

The dataset is dominated by ordinary cars: four cylinders (157 of 201), front-wheel drive (118),
overhead cam engines (145), naturally aspirated (165). Toyota alone contributes 32 cars, Nissan 18,
Mazda 17. The luxury brands that dominate the price tail contribute only a handful of rows each,
which is the tension running through the whole analysis. The cars that explain the top of the price
range are the ones with the least data behind them.

### 2.6 Correlation structure

In [ ]:
corr_cols = ["price", "engine-size", "curb-weight", "horsepower", "width", "length",
             "wheel-base", "bore", "city-mpg", "highway-mpg", "city-L/100km",
             "compression-ratio", "stroke", "peak-rpm", "height",
             "normalized-losses", "symboling"]

plt.figure(figsize=(12, 9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.75},
            annot_kws={"size": 8})
plt.title("Pearson correlation")
save_fig("06_correlation_heatmap")

In [ ]:
df[corr_cols].corr()["price"].drop("price").sort_values(key=abs, ascending=False).round(3)

Reading the first column of the heatmap: `engine-size` at 0.87, `curb-weight` at 0.83 and
`horsepower` at 0.81 are the three strongest links to price, followed by `width` at 0.75 and the
fuel economy columns at around -0.70.

The negative fuel economy correlations are not a causal story. Nobody pays more for a car because
it is thirsty; big engines are both expensive and inefficient, so economy is standing in for engine
size.

The more important thing in this heatmap is off the price row. `curb-weight` correlates 0.85 with
`engine-size`, 0.88 with `length` and 0.87 with `width`. The size and power columns are not seven
separate predictors, they are one underlying factor measured seven different ways. That has
consequences for Section 5, where I quantify it properly.

`height`, `peak-rpm`, `stroke`, `compression-ratio`, `symboling` and `normalized-losses` are all
under 0.14 in absolute terms, which is effectively nothing.

### 2.7 Summary of the patterns

Pulling Section 2 together:

1. **Price is right-skewed** (skew 1.81), mean 13,207 against a median of 10,295. A long luxury
   tail runs up to 45,400. Logging it would nearly fix the skew.
2. **Size and power dominate.** Engine size, kerb weight and horsepower each correlate above 0.80
   with price, and they all share the same right-skewed shape.
3. **Fuel economy is an echo of engine size**, not an independent driver. Its negative correlation
   with price is a side effect.
4. **The predictors are heavily entangled.** Weight, length, width and wheel-base correlate with
   each other at 0.78 to 0.88. Any statement about one of them is partly a statement about all.
5. **Geometry on its own is weak.** Height correlates 0.14 with price, so how tall a car is tells
   you almost nothing about what it costs.
6. **The insurance columns are dead.** `symboling` at -0.08 and `normalized-losses` at 0.13. In the
   case of `normalized-losses` that is partly self-inflicted, given that a fifth of it is a
   constant.
7. **The data is unbalanced.** Ordinary four-cylinder front-wheel-drive cars make up most of the
   rows, while the expensive cars that define the top of the price range are represented by a few
   examples each.

---

# 3. Grouping and average price &nbsp; *[5 Marks]*

> Group the data by identified parameters, such as drive-wheels and body-style. Find the average
> price for each group and present the results using visuals and statistics. Which groups have the
> highest and lowest prices?

### 3.1 Average price by drive-wheels

In [ ]:
dw = (df.groupby("drive-wheels")["price"]
        .agg(n="count", mean="mean", median="median", std="std", min="min", max="max")
        .sort_values("mean"))
dw.round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=dw.index, y=dw["mean"], order=dw.index, ax=axes[0],
            color="#4C72B0", errorbar=None)
for i, v in enumerate(dw["mean"]):
    axes[0].text(i, v + 400, f"{v:,.0f}", ha="center", fontsize=10)
axes[0].set_title("Mean price by drive-wheels")
axes[0].set_ylabel("price")

sns.boxplot(data=df, x="drive-wheels", y="price", order=dw.index, ax=axes[1])
axes[1].set_title("Full price distribution by drive-wheels")

plt.tight_layout()
save_fig("07_price_by_drive_wheels")

Rear-wheel drive averages 19,758 against 9,245 for front-wheel drive, which is more than double.
Four-wheel drive sits at 10,241 but on only 8 cars.

The boxplot is more honest than the bar chart. The rear-wheel-drive box is not just higher, it is
much wider, with a standard deviation of 9,083 against 3,346 for front-wheel drive. Rear-wheel
drive is not a price level, it is a marker for a segment that runs from an 8,000 Toyota up to a
45,000 Mercedes. The median for rear-wheel drive is 16,900, well below its mean of 19,758, so even
within that group the skew persists.

### 3.2 Average price by body-style

In [ ]:
bs = (df.groupby("body-style")["price"]
        .agg(n="count", mean="mean", median="median", std="std")
        .sort_values("mean"))
bs.round(0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=bs["mean"], y=bs.index, order=bs.index, ax=axes[0],
            color="#4C72B0", errorbar=None)
for i, (v, n) in enumerate(zip(bs["mean"], bs["n"])):
    axes[0].text(v + 300, i, f"{v:,.0f}  (n={n})", va="center", fontsize=9)
axes[0].set_title("Mean price by body-style")
axes[0].set_xlabel("price")
axes[0].set_xlim(0, 30000)

sns.boxplot(data=df, y="body-style", x="price", order=bs.index, ax=axes[1])
axes[1].set_title("Full price distribution by body-style")

plt.tight_layout()
save_fig("08_price_by_body_style")

The ordering is hardtop 22,209, convertible 21,891, saloon 14,460, estate 12,372, hatchback 9,957.

I have put the group sizes directly on the bars because they change the reading completely. The two
most expensive body styles are built on 8 and 6 cars. Their boxes are enormous: the hardtop
standard deviation is 14,556, which is larger than the mean price of a hatchback. The gap between
hatchback and saloon rests on 68 and 94 cars respectively and is far more trustworthy than anything
at the top of the chart.

### 3.3 Two-way grouping: drive-wheels against body-style

In [ ]:
pivot_mean = df.pivot_table(index="drive-wheels", columns="body-style",
                            values="price", aggfunc="mean")
pivot_n = df.pivot_table(index="drive-wheels", columns="body-style",
                         values="price", aggfunc="count")

print("Mean price")
display(pivot_mean.round(0))
print("\nNumber of cars")
display(pivot_n.fillna(0).astype(int))

In [ ]:
annot = pivot_mean.copy().astype("object")
for r in pivot_mean.index:
    for c in pivot_mean.columns:
        if pd.isna(pivot_mean.loc[r, c]):
            annot.loc[r, c] = "no\ncars"
        else:
            annot.loc[r, c] = f"{pivot_mean.loc[r, c]:,.0f}\n(n={int(pivot_n.loc[r, c])})"

plt.figure(figsize=(11, 4.5))
sns.heatmap(pivot_mean, annot=annot, fmt="", cmap="YlOrRd", linewidths=1,
            cbar_kws={"label": "mean price"}, annot_kws={"size": 10})
plt.title("Mean price by drive-wheels and body-style")
save_fig("09_pivot_heatmap")

This is the most informative table in Section 3. Read across the rear-wheel-drive row: 14,338 for a
hatchback up to 24,203 for a hardtop. Now read down the hatchback column: 7,603 for four-wheel
drive, 8,396 for front-wheel drive, 14,338 for rear-wheel drive.

The vertical spread is larger than the horizontal one. Within any body style, switching to
rear-wheel drive adds more than switching body style does within a drivetrain. That is the first
real evidence that `drive-wheels` is the stronger of the two variables, and it is consistent with
what the ANOVA says in 3.6.

The two empty cells are worth noting. There are no four-wheel-drive convertibles or hardtops in
this dataset, which is not a data problem, it is just what the market looked like. Several occupied
cells hold a single car, so the 11,595 for a front-wheel-drive convertible is one Volkswagen and
nothing more.

### 3.4 Other groupings worth looking at

In [ ]:
for col in ["num-of-cylinders", "engine-type", "aspiration", "engine-location",
            "num-of-doors", "fuel-system", "horsepower-binned"]:
    g = (df.groupby(col, observed=True)["price"]
           .agg(n="count", mean="mean", median="median")
           .sort_values("mean"))
    print(f"--- {col} ---")
    print(g.round(0).to_string())
    print()

In [ ]:
make_price = (df.groupby("make")["price"]
                .agg(n="count", mean="mean")
                .sort_values("mean"))

plt.figure(figsize=(10, 7))
sns.barplot(x=make_price["mean"], y=make_price.index, order=make_price.index,
            color="#4C72B0", errorbar=None)
for i, (v, n) in enumerate(zip(make_price["mean"], make_price["n"])):
    plt.text(v + 400, i, f"n={n}", va="center", fontsize=8, color="#555555")
plt.title("Mean price by make")
plt.xlabel("mean price")
plt.ylabel("")
save_fig("10_price_by_make")

`num-of-cylinders` produces the cleanest ordering of any variable in the dataset: four cylinders
10,303, five 22,008, six 23,672, eight 38,900. The two exceptions are the single three-cylinder
Chevrolet at the bottom and the four rotaries, which the "two cylinders" label describes badly
since a Wankel has rotors rather than cylinders.

`horsepower-binned` is almost as clean: 9,012, 15,913, 26,320 across Low, Medium and High, on group
sizes of 115, 62 and 24. That is a well-populated monotonic relationship.

`engine-location` shows the largest single gap in the dataset, 12,884 front against 34,528 rear.
It is also the least trustworthy number here, because those three rear-engined cars are Porsches.
What the variable has really captured is "is this a Porsche".

`aspiration` gives turbo 16,255 against naturally aspirated 12,542, a real but modest premium.
`num-of-doors` gives 13,498 against 12,818, which is nothing.

By make, Chevrolet is cheapest at 6,007 and Jaguar dearest at 34,600, but again on three cars each.

### 3.5 Highest and lowest priced groups

In [ ]:
rows = []
for col in ["drive-wheels", "body-style", "num-of-cylinders", "aspiration",
            "engine-location", "engine-type", "fuel-system", "num-of-doors",
            "horsepower-binned", "make"]:
    g = df.groupby(col, observed=True)["price"].agg(["count", "mean"])
    hi, lo = g["mean"].idxmax(), g["mean"].idxmin()
    rows.append({
        "variable": col,
        "cheapest": f"{lo} ({g.loc[lo, 'mean']:,.0f}, n={int(g.loc[lo, 'count'])})",
        "dearest": f"{hi} ({g.loc[hi, 'mean']:,.0f}, n={int(g.loc[hi, 'count'])})",
        "ratio": round(g["mean"].max() / g["mean"].min(), 2),
    })

pd.DataFrame(rows).sort_values("ratio", ascending=False).reset_index(drop=True)

Answering the question directly, and restricting myself to groups with enough cars to mean
something:

- **Dearest group:** rear-wheel-drive hardtops at 24,203, or rear-wheel drive generally at 19,758.
  By engine, eight cylinders at 38,900, though that is four cars.
- **Cheapest group:** front-wheel-drive hatchbacks at 8,396, or hatchbacks generally at 9,957.
  By engine, four cylinders at 10,303.

The `ratio` column ranks the variables by how far apart their extremes sit, and the top of that
ranking is dominated by the small-sample variables (`make`, `fuel-system`, `num-of-cylinders`,
`engine-location`). A large ratio between two groups of three cars is not evidence of much. The
ANOVA in the next subsection is a better guide.

### 3.6 Are those group differences real?

In [ ]:
results = []
for col in CATEGORICAL:
    groups = [g["price"].values for _, g in df.groupby(col, observed=True) if len(g) > 1]
    f_stat, p_val = stats.f_oneway(*groups)

    grand = df["price"].mean()
    ss_between = sum(len(g) * (g["price"].mean() - grand) ** 2
                     for _, g in df.groupby(col, observed=True))
    ss_total = ((df["price"] - grand) ** 2).sum()

    results.append({"variable": col, "levels": df[col].nunique(),
                    "F": round(f_stat, 2), "p": p_val,
                    "eta_squared": round(ss_between / ss_total, 3)})

anova = pd.DataFrame(results).sort_values("eta_squared", ascending=False).reset_index(drop=True)
anova["p"] = anova["p"].map(lambda v: f"{v:.2e}")
anova

Eta squared is the share of the variance in price that the variable accounts for, so it is directly
comparable across rows in a way the F statistic is not.

`make` tops the table at 0.796, meaning brand alone accounts for roughly 80% of the variation in
price. That is both true and nearly useless: 22 levels fitted on 201 cars will absorb variance
almost by construction, and it tells a dealer only that Jaguars cost more than Chevrolets.

The result I would actually rely on is `drive-wheels` at 0.407 with three well-populated levels,
and `horsepower-binned` at 0.523. `body-style` reaches only 0.157 despite the dramatic bar chart in
3.2, and `num-of-doors` at 0.002 with p = 0.55 is the one categorical variable in the dataset with
no detectable relationship to price at all.

In [ ]:
import itertools


def pairwise(col):
    levels = sorted(df[col].unique())
    combos = list(itertools.combinations(levels, 2))
    out = []
    for a, b in combos:
        x, y = df.loc[df[col] == a, "price"], df.loc[df[col] == b, "price"]
        t_stat, p_val = stats.ttest_ind(x, y, equal_var=False)
        out.append({"group A": a, "group B": b, "n A": len(x), "n B": len(y),
                    "mean gap": round(x.mean() - y.mean()),
                    "p": round(p_val, 5),
                    "survives Bonferroni": p_val * len(combos) < 0.05})
    return pd.DataFrame(out)


print("drive-wheels")
display(pairwise("drive-wheels"))
print("body-style")
display(pairwise("body-style"))

The ANOVA says the groups differ somewhere; these pairwise tests say where, with a Bonferroni
correction so that running ten tests does not manufacture a result.

For `drive-wheels`, rear-wheel drive is significantly dearer than both front-wheel drive
(gap 10,513) and four-wheel drive (gap 9,517), and both survive the correction comfortably.
Four-wheel versus front-wheel drive is a gap of 996 with p = 0.43, so there is no evidence those
two differ.

For `body-style`, only hatchback versus saloon survives, a gap of 4,502 on 68 and 94 cars. Every
comparison involving convertibles or hardtops fails once corrected, despite gaps of 10,000 or more,
because 6 and 8 cars with standard deviations above 11,000 cannot support that claim. This is the
clearest example in the notebook of a difference that looks enormous on a bar chart and evaporates
under a test, and it is why I put the group sizes on the chart in 3.2.

---

# 4. Effect of the independent attributes on price &nbsp; *[5 Marks]*

> Identify the effect of the independent attributes on price using visuals and statistics.

## 4a. Continuous variables against price &nbsp; *[2 Marks]*

In [ ]:
scatter_cols = ["engine-size", "curb-weight", "horsepower", "width",
                "highway-mpg", "wheel-base", "bore", "compression-ratio", "height"]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flat, scatter_cols):
    sns.regplot(data=df, x=col, y="price", ax=ax,
                scatter_kws={"s": 18, "alpha": 0.55},
                line_kws={"color": "#C44E52", "lw": 2})
    r, p = stats.pearsonr(df[col], df["price"])
    ax.set_title(f"{col}   r = {r:.2f}", fontsize=11)
    ax.set_ylabel("price" if ax in axes[:, 0] else "")
plt.tight_layout()
save_fig("11_continuous_vs_price")

In [ ]:
rows = []
for col in CONTINUOUS + ["symboling", "diesel"]:
    r, p = stats.pearsonr(df[col], df["price"])
    rho, _ = stats.spearmanr(df[col], df["price"])
    rows.append({"variable": col, "pearson r": round(r, 3), "spearman rho": round(rho, 3),
                 "r squared": round(r ** 2, 3), "p value": p,
                 "significant": "yes" if p < 0.05 else "no"})

corr_tbl = pd.DataFrame(rows).sort_values("r squared", ascending=False).reset_index(drop=True)
corr_tbl["p value"] = corr_tbl["p value"].map(lambda v: f"{v:.2e}")
corr_tbl

Nine of the sixteen continuous variables are significantly related to price at the 5% level, and
the ones that are not (`height`, `normalized-losses`, `diesel`, `peak-rpm`, `symboling`, `stroke`,
`compression-ratio`) are all below 0.14 in absolute correlation, so their non-significance is not a
borderline call.

Reading the scatter panels rather than just the numbers:

- **`engine-size`, r = 0.87.** The strongest linear relationship in the dataset. Every additional
  cubic inch shows up in the price, and the points hug the line until about 200 cubic inches, after
  which the handful of large engines fan out widely. A single engine size of 200+ can mean anything
  from 22,000 to 45,000.
- **`curb-weight`, r = 0.83.** Nearly as strong and visibly tighter at the bottom end. Weight is a
  decent proxy for how much car you are buying.
- **`horsepower`, r = 0.81.** Strong, but with a visible curve. The gap between 50 and 100 hp costs
  far less than the gap between 150 and 200 hp, which suggests the true relationship is closer to
  exponential than linear.
- **`width`, r = 0.75.** Stronger than I expected for a dimension that varies so little. Wide cars
  are premium cars.
- **`highway-mpg`, r = -0.71.** The strongest negative. As established in 2.6, this is engine size
  seen from the other side.
- **`wheel-base`, r = 0.59 and `bore`, r = 0.54.** Real but moderate.
- **`compression-ratio`, r = 0.07 and `height`, r = 0.14.** Flat scatter clouds, no relationship.
  The compression-ratio panel shows the two-population split clearly: a vertical band near 9 and a
  second near 22, with no slope between them.

The Spearman column is worth a second look. For `horsepower` the rank correlation is 0.85 against
a Pearson value of 0.81, and for `curb-weight` 0.91 against 0.83. Both are measuring a relationship
that is reliably increasing but not quite straight, which is the curvature showing up. `engine-size`
goes the other way, 0.83 against 0.87, so it is the one variable here whose relationship with price
really is closest to linear.

## 4b. Categorical variables against price &nbsp; *[2 Marks]*

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, ["drive-wheels", "horsepower-binned", "num-of-cylinders",
                               "aspiration", "engine-type", "body-style"]):
    order = df.groupby(col, observed=True)["price"].median().sort_values().index
    sns.boxplot(data=df, x=col, y="price", order=order, ax=ax)
    sns.stripplot(data=df, x=col, y="price", order=order, ax=ax,
                  color="black", size=2.5, alpha=0.4)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
save_fig("12_categorical_vs_price")

The individual cars are overlaid as dots so the thin groups are visible rather than hidden behind a
box. The twelve-cylinder level in `num-of-cylinders` is a single dot; the rotary level is four.

In [ ]:
plt.figure(figsize=(9, 5))
eta_order = anova.sort_values("eta_squared")
sns.barplot(x=eta_order["eta_squared"], y=eta_order["variable"], color="#4C72B0", errorbar=None)
for i, (v, lv) in enumerate(zip(eta_order["eta_squared"], eta_order["levels"])):
    plt.text(v + 0.008, i, f"{v:.3f}  ({lv} levels)", va="center", fontsize=9)
plt.xlim(0, 1.0)
plt.title("Share of price variance explained by each categorical variable (eta squared)")
plt.xlabel("eta squared")
plt.ylabel("")
save_fig("13_eta_squared")

This chart reuses the ANOVA from 3.6, sorted so the ranking is easier to read, with the number of
levels annotated because eta squared rises mechanically with more levels.

Taking the variables in order of how seriously I take them:

- **`horsepower-binned`, eta squared 0.523 on 3 levels.** The strongest honest categorical effect.
  Three well-populated groups, a monotonic ordering, and a clean mechanism.
- **`drive-wheels`, 0.407 on 3 levels.** Backed by the pairwise tests in 3.6. Rear-wheel drive is
  genuinely and substantially dearer.
- **`num-of-cylinders`, 0.630 on 7 levels.** The ordering is clean but two of the seven levels hold
  one car each, so the headline figure is inflated.
- **`make`, 0.796 on 22 levels.** Highest in the table and least usable. With 22 levels across 201
  cars it is close to fitting an indicator per brand.
- **`fuel-system`, 0.353 on 8 levels.** Mostly a restatement of engine size, since `mpfi` is fitted
  to the larger engines and the carburettor systems to the small ones.
- **`body-style`, 0.157 on 5 levels.** Weaker than the Section 3 bar chart suggested, for the
  small-sample reason set out in 3.6.
- **`engine-location`, 0.110 on 2 levels.** Three rear-engined Porsches.
- **`aspiration`, 0.032.** Small but real, p = 0.011.
- **`num-of-doors`, 0.002, p = 0.55.** No effect. Whether a car has two doors or four tells you
  nothing about its price.

## 4c. Two categorical variables &nbsp; *[1 Mark]*

Two different questions can be asked about a pair of categorical variables. First, are they
associated with each other? Second, do they combine to affect price in a way that neither explains
alone? I take them in that order.

In [ ]:
contingency = pd.crosstab(df["drive-wheels"], df["body-style"])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
cramers_v = np.sqrt(chi2 / (contingency.values.sum() * (min(contingency.shape) - 1)))

print(f"chi squared      : {chi2:.2f}")
print(f"degrees of freedom: {dof}")
print(f"p value          : {p_val:.3e}")
print(f"Cramers V        : {cramers_v:.3f}")
print(f"expected counts below 5: {(expected < 5).sum()} of {expected.size} cells")
contingency

The test says the two variables are not independent (p = 0.0003), and Cramér's V of 0.27 puts the
strength of that association at moderate. Looking at the table, rear-wheel drive accounts for 5 of
6 convertibles and 7 of 8 hardtops, while front-wheel drive takes 49 of 68 hatchbacks. Sports and
luxury body styles come with rear-wheel drive; economy body styles come with front-wheel drive.

The caveat matters though: 9 of the 15 expected counts are below 5, which breaks the usual
assumption behind the chi-squared approximation. The association is almost certainly real given how
lopsided the table is, but I would not quote that p value precisely.

This also explains something from Section 3. Part of the reason hardtops and convertibles looked so
expensive is that they are almost all rear-wheel drive. The two variables are measuring overlapping
things.

In [ ]:
print("Mean price")
display(df.pivot_table(index="drive-wheels", columns="aspiration",
                       values="price", aggfunc="mean").round(0))
print("Number of cars")
display(df.pivot_table(index="drive-wheels", columns="aspiration",
                       values="price", aggfunc="count"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=df, x="drive-wheels", y="price", hue="aspiration",
            order=["fwd", "4wd", "rwd"], ax=axes[0], errorbar=None)
axes[0].set_title("Mean price by drive-wheels and aspiration")

sns.pointplot(data=df, x="drive-wheels", y="price", hue="aspiration",
              order=["fwd", "4wd", "rwd"], ax=axes[1], errorbar=None,
              markers=["o", "s"], linestyles=["-", "--"])
axes[1].set_title("Interaction plot: do the lines stay parallel?")

plt.tight_layout()
save_fig("14_two_categorical_interaction")

Here is the interaction, and it is a genuinely interesting one. The turbocharger premium is not
constant:

| Drive wheels | Naturally aspirated | Turbo | Premium |
|---|---|---|---|
| fwd | 8,724 (n=102) | 12,565 (n=16) | **+3,841** |
| 4wd | 9,829 (n=6) | 11,476 (n=2) | +1,647 |
| rwd | 19,660 (n=57) | 20,066 (n=18) | **+406** |

On a front-wheel-drive car a turbo adds nearly 3,900 to the average price. On a rear-wheel-drive
car it adds almost nothing. The lines in the interaction plot are nowhere near parallel, which is
the visual signature of exactly this.

The explanation is about what the turbo is for in each segment. On a front-driven economy car the
turbo is the performance option, and it is what distinguishes the sporty version from the base
model. On a rear-driven car the price is already set by a large engine, and forced induction is one
feature among several on a car that was expensive to begin with.

The practical point is that the effect of aspiration cannot be stated as a single number. It
depends on what the car already is, which is precisely what an interaction means.

---

# 5. Main price drivers and a recommendation &nbsp; *[5 Marks]*

> Based on your findings, identify the main variables that have the most impact on car price, and
> support your answer with a feature selection method. Write a short recommendation for a car
> dealer setting prices.

Sections 2 to 4 each gave a partial answer: correlation favours `engine-size`, eta squared favours
`make`, and the pairwise tests favour `drive-wheels`. Rather than pick whichever one suits the
conclusion, I run four different feature selection methods and look at where they agree.

### 5.1 Building the modelling matrix

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import f_regression, mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

# Excluded on purpose, per Section 1.7:
#   city-L/100km  -> exactly 235 / city-mpg
#   city-mpg      -> keeping highway-mpg only, the two correlate at 0.97
#   gas           -> gas = 1 - diesel
MODEL_NUM = ["wheel-base", "length", "width", "height", "curb-weight", "engine-size",
             "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
             "highway-mpg", "symboling", "normalized-losses", "diesel"]

MODEL_CAT = ["drive-wheels", "aspiration", "body-style", "engine-type",
             "num-of-cylinders", "fuel-system", "engine-location", "num-of-doors"]

X = pd.get_dummies(df[MODEL_NUM + MODEL_CAT], drop_first=True).astype(float)
y = df["price"]

print(f"{X.shape[1]} features for {X.shape[0]} cars")
print("make is deliberately left out, see the note below")

Two deliberate exclusions beyond the redundant columns.

`city-mpg` goes because it correlates with `highway-mpg` at 0.97; keeping both would split one
effect across two features and halve both importance scores.

`make` goes because of what it would do to the results. It would one-hot into 21 extra columns and,
on the strength of its eta squared of 0.796, absorb most of the explanatory power. The model would
then conclude that the main driver of price is being a Jaguar, which is true and unhelpful. I want
to know which *measurable attributes* of a car set its price, and brand is the thing those
attributes are supposed to explain. I come back to it in the recommendation.

### 5.2 Method 1: correlation with price

In [ ]:
corr_scores = X.apply(lambda col: abs(np.corrcoef(col, y)[0, 1]))
corr_scores.sort_values(ascending=False).head(12).round(3)

The simplest method, and the most limited: it only sees straight-line relationships and judges each
feature in isolation. `engine-size` leads at 0.872.

### 5.3 Method 2: ANOVA F-test (SelectKBest)

In [ ]:
f_scores, f_pvalues = f_regression(X, y)
f_series = pd.Series(f_scores, index=X.columns)
f_table = pd.DataFrame({"F": f_scores, "p": f_pvalues}, index=X.columns)
f_table.sort_values("F", ascending=False).head(12).assign(
    p=lambda d: d["p"].map(lambda v: f"{v:.2e}")).round(2)

`f_regression` is what `SelectKBest` uses for a continuous target. For a single predictor it is
monotonically related to the correlation, so the ordering here largely repeats 5.2, but it adds p
values, which tell us how confident we can be in each. `engine-size` has F = 634 on 1 and 199
degrees of freedom, which is overwhelming.

### 5.4 Method 3: mutual information

In [ ]:
mi_scores = pd.Series(
    mutual_info_regression(X, y, random_state=RANDOM_STATE),
    index=X.columns)
mi_scores.sort_values(ascending=False).head(12).round(3)

Mutual information measures any statistical dependence, not just linear, so it is the method that
can see the curved horsepower relationship noted in 4a.

That shows up immediately. `curb-weight` moves ahead of `engine-size`, and `horsepower` and
`highway-mpg` both score higher than their correlations would suggest. The ranking is otherwise
similar, which is reassuring: the relationships here are mostly monotonic even where they are not
perfectly linear.

### 5.5 Method 4: random forest importance

In [ ]:
rf = RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE,
                           oob_score=True, n_jobs=-1)
rf.fit(X, y)

print(f"out-of-bag R squared : {rf.oob_score_:.3f}")
print(f"training R squared   : {rf.score(X, y):.3f}")

rf_scores = pd.Series(rf.feature_importances_, index=X.columns)
rf_scores.sort_values(ascending=False).head(12).round(4)

The out-of-bag R squared of 0.93 means the forest predicts the price of cars it has not seen to
within 7% of the total variance, so the importance scores rest on a model that actually works.

This is the first method that is multivariate: it judges each feature given all the others, not in
isolation. The effect is dramatic. `engine-size` takes 0.64 of the total importance and
`curb-weight` 0.21, between them 85%, and everything else is left with scraps. `horsepower` falls
to 0.016 despite correlating 0.81 with price.

That collapse is not evidence that horsepower is irrelevant. It is evidence that horsepower tells
the model nothing it has not already learned from engine size, with which it correlates at 0.82.
Tree ensembles split correlated features arbitrarily and the one that happens to be chosen first
takes the credit. This is the single most important caveat in Section 5, and it is why I do not
rely on this method alone.

In [ ]:
perm = permutation_importance(rf, X, y, n_repeats=20,
                              random_state=RANDOM_STATE, n_jobs=-1)
pd.Series(perm.importances_mean, index=X.columns) \
  .sort_values(ascending=False).head(10).round(4)

Permutation importance measures how much worse the model gets when a feature is shuffled. It
confirms the same story and has the same blind spot: shuffling `horsepower` barely hurts, because
`engine-size` is still there to carry the information.

### 5.6 Putting the four methods together

In [ ]:
ranks = pd.DataFrame({
    "correlation": corr_scores,
    "F score": f_series,
    "mutual info": mi_scores,
    "random forest": rf_scores,
})

for col in ranks.columns:
    ranks[f"rank: {col}"] = ranks[col].rank(ascending=False)

rank_cols = [c for c in ranks.columns if c.startswith("rank:")]
ranks["mean rank"] = ranks[rank_cols].mean(axis=1)

top15 = ranks.sort_values("mean rank").head(15)
top15[["correlation", "F score", "mutual info", "random forest", "mean rank"]].round(3)

In [ ]:
plot_df = top15.sort_values("mean rank", ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(x=plot_df["mean rank"], y=plot_df.index, color="#4C72B0", errorbar=None)
plt.title("Consolidated feature ranking across four selection methods\n(lower is better)")
plt.xlabel("mean rank across correlation, F-test, mutual information and random forest")
plt.ylabel("")
save_fig("15_consolidated_ranking")

All four methods agree on the top of the list, which is the result I was looking for. The ordering
is `engine-size` and `curb-weight` tied at a mean rank of 1.75, then `horsepower` at 3.25,
`highway-mpg` at 3.75 and `width` at 4.5. After that come `length` at 7.25, `wheel-base` at 8.0 and
`bore` at 10.25, and the first categorical feature to appear is `num-of-cylinders_four` at 11.0,
followed by `drive-wheels_rwd` at 12.75.

So the measurable attributes that drive price are, in order: **how big the engine is, how heavy the
car is, how much power it makes, and how wide it is.** Drivetrain and cylinder count matter, but
they rank behind all four of those, and they are partly describing the same cars.

### 5.7 How much of this is really one variable?

In [ ]:
top_features = ["engine-size", "curb-weight", "horsepower", "width",
                "length", "wheel-base", "highway-mpg"]

corr_matrix = df[top_features].corr()
vif = pd.Series(np.diag(np.linalg.inv(corr_matrix.values)), index=top_features)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation among the top-ranked features")
save_fig("16_top_feature_collinearity")

vif.sort_values(ascending=False).round(2).to_frame("VIF")

This is the caveat that governs how the whole of Section 5 should be read.

The seven top-ranked features correlate with each other at between 0.37 and 0.88. The variance
inflation factors run from 4.2 to 12.1, and anything above 5 is normally treated as a problem,
above 10 as serious. `curb-weight` at 12.1 is the worst.

They are not seven independent price drivers. They are one latent factor, which I would call
"how much car is there", measured seven different ways. A car with a big engine is heavy, wide,
long and thirsty, and it rides on a long wheelbase. That is why the random forest could hand 85% of
its importance to two features without losing accuracy, and why `horsepower` can correlate 0.81
with price and still appear to be worth nothing once `engine-size` is in the model.

The practical consequence: the top of the ranking identifies the right *factor*, but the ordering
within it is partly arbitrary. I would not claim that engine size matters more than kerb weight on
the basis of a mean rank of 1.75 against 1.75.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def cv_r2(cols):
    return cross_val_score(LinearRegression(), X[cols], y, cv=cv, scoring="r2").mean()


comparison = pd.DataFrame([
    {"model": "engine-size alone", "features": 1,
     "cv R squared": cv_r2(["engine-size"])},
    {"model": "engine-size + curb-weight + horsepower", "features": 3,
     "cv R squared": cv_r2(["engine-size", "curb-weight", "horsepower"])},
    {"model": "engine-size + curb-weight + horsepower + width", "features": 4,
     "cv R squared": cv_r2(["engine-size", "curb-weight", "horsepower", "width"])},
    {"model": "all features, linear", "features": X.shape[1],
     "cv R squared": cross_val_score(LinearRegression(), X, y, cv=cv, scoring="r2").mean()},
    {"model": "all features, random forest", "features": X.shape[1],
     "cv R squared": cross_val_score(
         RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
         X, y, cv=cv, scoring="r2").mean()},
])
comparison.round(3)

A sanity check on whether the shortlist is worth anything, using five-fold cross-validation so the
scores reflect unseen cars.

`engine-size` on its own reaches an R squared of 0.68. Adding kerb weight and horsepower takes it
to 0.74, and width to 0.76. Throwing in all 42 features gets a linear model to 0.82 and a random
forest to 0.88.

So four variables recover about 86% of the accuracy that all forty-two achieve. The remaining gap
is real but it costs ten times the complexity to close, and most of it comes from the forest
handling the curvature in horsepower rather than from any new variable. For a dealer wanting a
rule of thumb rather than a model, four variables is the sensible stopping point.

### 5.8 Recommendation for a car dealer setting prices

**The short version.** Four measurements explain roughly three quarters of what a car in this
market is worth: engine size, kerb weight, horsepower and width. Start from those, adjust for
drivetrain and brand, and ignore most of the rest.

**1. Price from engine size first.** It is the strongest single predictor in every method tested
(r = 0.87, and 0.68 R squared on its own). As a rough guide, each extra 10 cubic inches of
displacement is worth somewhere around 1,500 in this dataset. If you can record only one
specification per car, record this one.

**2. Treat size, weight and power as one adjustment, not three.** They correlate with each other at
0.76 to 0.88 (Section 5.7). A heavy, wide, powerful car with a big engine is a single kind of car,
and adding a separate premium for each of those attributes will overprice it. Decide where the car
sits on one "how much car is this" scale and price that once.

**3. Charge a real premium for rear-wheel drive.** Rear-wheel-drive cars average 19,758 against
9,245 for front-wheel drive, a gap that survives correction for multiple testing (p < 0.001).
Within any body style, switching to rear-wheel drive adds more than switching body style does
within a drivetrain. Four-wheel drive does **not** command a premium over front-wheel drive here
(gap 996, p = 0.43), so do not price it as though it does.

**4. Price the turbo differently depending on the car.** On front-wheel-drive stock a turbo is
worth about 3,800 on average, because it is what separates the sporty version from the base model.
On rear-wheel-drive stock it is worth about 400, because the price is already set by the engine.
Applying one flat turbo uplift across the lot will overprice half the inventory and underprice the
other half.

**5. Stop collecting some things.** Number of doors has no measurable relationship to price at all
(eta squared 0.002, p = 0.55). Height, compression ratio, peak rpm, stroke and the insurance
`symboling` score are all below 0.14 in correlation. None of these should influence a price.

**6. Brand is the largest single factor, and it is not a specification.** Make alone accounts for
about 80% of price variance, from Chevrolet at 6,007 to Jaguar at 34,600. I kept it out of the
model on purpose, because "it is a Jaguar" explains a price without giving you a method. In
practice it means the four-variable rule should be applied *within* a brand tier rather than across
the whole lot. A Mercedes and a Toyota with identical engines will not and should not fetch the
same price.

**7. Know where the rule stops working.** Below 20,000 these relationships are tight. Above it they
spread out badly: cars with engines over 200 cubic inches range from 22,000 to 45,000 on
specification alone, and Section 1.5 found seven pairs of mechanically identical cars whose prices
differ by up to 3,000. At the luxury end, specification sets a floor and trim, equipment and brand
decide the rest. Price those individually rather than from a formula.

**Caveats worth stating plainly.** This is 201 cars from a single late-1980s snapshot, so the
coefficients are of their time even though the structure probably is not. Several conclusions rest
on very few cars: 3 rear-engined, 6 convertibles, 8 hardtops, and one car each in the three- and
twelve-cylinder classes. And everything above is association, not causation. Fitting a bigger
engine to a Chevrolet would not turn it into a Jaguar.

---

## Conclusion

The question was which variables have the most impact on car price. The answer that four
independent selection methods agree on is **engine size, kerb weight, horsepower and width**, with
drive-wheels and cylinder count as meaningful secondary effects, and brand sitting above all of
them as something the specifications are trying to explain.

The three findings I did not expect going in:

- Most of what the IQR rule flagged as outliers was real structure. The 27 compression-ratio
  outliers were the diesel population, and the 14 price outliers were the luxury cars that make the
  whole question interesting.
- `body-style` looked like a strong price driver on a bar chart and mostly failed the pairwise
  tests. The two dearest body styles are built on 6 and 8 cars.
- The turbo premium depends entirely on drivetrain, 3,841 on front-wheel drive against 406 on
  rear-wheel drive. That interaction is invisible to any analysis that looks at one variable at a
  time.

The honest limitation is the one in 5.7: the top-ranked features are not independent of each other.
They measure a single underlying factor from different angles, and no feature selection method run
on this dataset can separate them, because in the real world a car with a big engine simply is
heavy, wide and powerful.